NOTE: this does not work out of the box. Mummichog is a bit annoying to run. The latest update introduced a bug so you have to use version 2.6.1. **It also only works on Python 3.8.**\
If you have `pyenv` you can do `pyenv install 3.8 && pyenv global 3.8 && pip install mummichog==2.6.1`.
Unfortunately Terra does not have Python 3.8 so this must be run locally.

# Mummichog
## Install

In [ ]:
# Remember, must be using Python 3.8 in order for it to work...
system('pip install mummichog==2.6.1')

## Lib

In [ ]:
library(data.table)
dir.create('scratch/mummichog_input', recursive=T)
dir.create('scratch/mummichog_output',recursive=T)

## Data

In [ ]:
GxMpGxE_models <- fread('GxM_results.csv')
met_info <- fread('data/derived/metabolomics/met_info-mesa.csv')

## Generate inputs
Creates a mummichog input file for every Y x E x G combination. Written to `scratch/mummichog_inputs/`.

In [ ]:
met_info[
  ][, .(unique_met_id,MZ,RT)
  ][GxMpGxE_models[,.(Y,E,G,M,p,t)],
    on=.(unique_met_id=M)
  ][!is.na(MZ) & !is.na(RT)
    , by=.(Y,E,G)
    , .(MZ,RT,p,t,M=unique_met_id) |>
      fwrite(sep='\t', file=paste0(
        'scratch/mummichog_input/mummichog_input-',
        .BY$G,'-',.BY$Y,'-',.BY$E,'.tsv')
      )
]

## Run
Outputs go in `scratch/mummichog_output/`.

In [ ]:
fs <- list.files('scratch/mummichog_input/',full.names=T)
t0 <- Sys.time()
lapply(fs, \(f) {
  print(f)
  system(paste('mummichog -f', f, '-o', basename(f)))

  # Hacky stuff to move mummichog's strangely-named outputs to where I want
  out_dirnm <- sub('mummichog_input-','',sub('.csv$','',basename(f)))
  out_dirnm2 <- paste0('scratch/mummichog_output/',out_dirnm)
  unlink(out_dirnm2,recursive=T)
  file.rename(list.files(pattern=out_dirnm), out_dirnm2)
})
Sys.time()-t0